# Телеграм-бот с функциями распознавания аудио и видео сообщения в текст, а также озвучку текстовых сообщений, отправленных в бот

## Install

In [ ]:
!pip install SpeechRecognition

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 19.3 MB/s eta 0:00:00


In [ ]:
!pip install aiogram

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 666.6/666.6 kB 6.3 MB/s eta 0:00:00


In [ ]:
import logging
import speech_recognition as sr
from aiogram import Bot, Dispatcher, types

import asyncio
import logging
import sys
from os import getenv

from aiogram import Bot, Dispatcher, Router, types
from aiogram.enums import ParseMode
from aiogram.filters import CommandStart
from aiogram.types import Message
from aiogram.utils.markdown import hbold

## Echo Bot

In [ ]:
# Bot token can be obtained via https://t.me/BotFather
TOKEN = getenv("TOKEN")

# All handlers should be attached to the Router (or Dispatcher)
dp = Dispatcher()


@dp.message(CommandStart())
async def command_start_handler(message: Message) -> None:
    """
    This handler receives messages with `/start` command
    """
    # Most event objects have aliases for API methods that can be called in events' context
    # For example if you want to answer to incoming message you can use `message.answer(...)` alias
    # and the target chat will be passed to :ref:`aiogram.methods.send_message.SendMessage`
    # method automatically or call API method directly via
    # Bot instance: `bot.send_message(chat_id=message.chat.id, ...)`
    await message.answer(f"Hello, {hbold(message.from_user.full_name)}!")


@dp.message()
async def echo_handler(message: types.Message) -> None:
    """
    Handler will forward receive a message back to the sender

    By default, message handler will handle all message types (like a text, photo, sticker etc.)
    """
    try:
        # Send a copy of the received message
        await message.send_copy(chat_id=message.chat.id)
    except TypeError:
        # But not all the types is supported to be copied so need to handle it
        await message.answer("Nice try!")


async def main() -> None:
    # Initialize Bot instance with a default parse mode which will be passed to all API calls
    bot = Bot("6570481069:AAEhIll4fHnIPQOQMRd3jutV4oLaiVUuaiw", parse_mode=ParseMode.HTML)
    # And the run events dispatching
    await dp.start_polling(bot)


def the_main():
    logging.basicConfig(level=logging.INFO, stream=sys.stdout)
    asyncio.run(main())


import nest_asyncio
nest_asyncio.apply()


/usr/lib/python3.10/inspect.py:1301: RuntimeWarning: coroutine 'main' was never awaited
  for param in sig.parameters.values():


In [ ]:
the_main()

## Echo с разной обработкой аудио, видео, текста

In [ ]:
import os

In [ ]:
import asyncio
import logging
import sys
from os import getenv

from aiogram import Bot, Dispatcher, Router, types
from aiogram.enums import ParseMode
from aiogram.filters import CommandStart
from aiogram.types import Message
from aiogram.utils.markdown import hbold

# Bot token can be obtained via https://t.me/BotFather
TOKEN = getenv("TOKEN")

# All handlers should be attached to the Router (or Dispatcher)
dp = Dispatcher()


@dp.message(CommandStart())
async def command_start_handler(message: Message) -> None:
    """
    This handler receives messages with `/start` command
    """
    # Most event objects have aliases for API methods that can be called in events' context
    # For example if you want to answer to incoming message you can use `message.answer(...)` alias
    # and the target chat will be passed to :ref:`aiogram.methods.send_message.SendMessage`
    # method automatically or call API method directly via
    # Bot instance: `bot.send_message(chat_id=message.chat.id, ...)`
    await message.answer(f"Hello, {hbold(message.from_user.full_name)}!")


@dp.message()
async def echo_handler(message: types.Message) -> None:
    """
    Handler will forward receive a message back to the sender

    By default, message handler will handle all message types (like a text, photo, sticker etc.)
    """
    try:
        # Send a copy of the received message
        if message.content_type == types.ContentType.AUDIO:
          message.answer("Audio!")
        elif message.content_type == types.ContentType.VIDEO:
          message.answer("Video!")
        elif message.content_type == types.ContentType.TEXT:
          message.answer("TEXT!")
        else:
          message.answer("Something")
        await message.send_copy(chat_id=message.chat.id)
    except TypeError:
        # But not all the types is supported to be copied so need to handle it
        await message.answer("Nice try!")


async def main() -> None:
    # Initialize Bot instance with a default parse mode which will be passed to all API calls
    bot = Bot("TOKEN", parse_mode=ParseMode.HTML)
    # And the run events dispatching
    await dp.start_polling(bot)


def the_main():
    logging.basicConfig(level=logging.INFO, stream=sys.stdout)
    asyncio.run(main())


import nest_asyncio
nest_asyncio.apply()

In [ ]:
the_main()

## Рабочая версия с распознаванием аудио и видио в текст

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import os
import speech_recognition as sr
import asyncio
import logging
import sys
from os import getenv

from aiogram import Bot, Dispatcher, Router, F
from aiogram.enums import ParseMode
from aiogram.filters import CommandStart
from aiogram.types import Message

TOKEN = "TOKEN"

bot = Bot(TOKEN)
dp = Dispatcher()

@dp.message()
async def echo_handler(message: types.Message):
    if message.content_type == types.ContentType.VOICE:
        voice_path = "voice.ogg"
        await bot.download(message.voice, destination=voice_path)

        os.system('ffmpeg -y -i voice.ogg voice.wav')

        # Создание объекта Recognizer из библиотеки SpeechRecognition
        recognizer = sr.Recognizer()

        # Распознавание речи в голосовом сообщении
        try:
            with sr.AudioFile('voice.wav') as source:
                audio_data = recognizer.record(source)
                text = recognizer.recognize_google(audio_data, language="ru-RU")  #  нужный язык
                await message.reply(text, parse_mode=ParseMode.MARKDOWN)
        except sr.UnknownValueError:
            await message.reply("Не удалось распознать речь в голосовом сообщении.")
        except sr.RequestError as e:
            await message.reply(f"Ошибка при запросе к сервису распознавания речи: {str(e)}")

    elif message.content_type == types.ContentType.VIDEO_NOTE:
      video_path = 'video.mp4'
      await bot.download(message.video_note, destination=video_path)

      os.system('ffmpeg -y -i video.mp4 video.wav')

      recognizer = sr.Recognizer()

      try:
        with sr.AudioFile('video.wav') as source:
          audio_data = recognizer.record(source)
          text = recognizer.recognize_google(audio_data, language="ru-RU")  #  нужный язык
          await message.reply(text, parse_mode=ParseMode.MARKDOWN)
      except sr.UnknownValueError:
        await message.reply("Не удалось распознать речь в видео.")
      except sr.RequestError as e:
        await message.reply(f"Ошибка при запросе к сервису распознавания речи: {str(e)}")

        await message.answer("Видео кружок!")


async def main() -> None:
    await dp.start_polling(bot, skip_updates=True)

asyncio.run(main())

## Последняя рабочая версия с функциями распознавания video/audio + text_to_voice

In [ ]:
!pip install -q torchaudio omegaconf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [ ]:
import torch

In [ ]:
# выбор языка озвучки и id модели
language = 'ru'
model_id = 'v3_1_ru'

# загрузка модели с torchhub
model, example_text = torch.hub.load(repo_or_dir='snakers4/silero-models',
                                     model='silero_tts',
                                     language=language,
                                     speaker=model_id,
                                     trust_repo=True)

# список доступных голосов
model.speakers

Downloading: "https://github.com/snakers4/silero-models/zipball/master" to /root/.cache/torch/hub/master.zip
100%|██████████| 59.0M/59.0M [00:02<00:00, 20.8MB/s]


['aidar', 'baya', 'kseniya', 'xenia', 'eugene', 'random']

In [ ]:
%%time
from IPython.display import Audio, display

# частота дискретизации
sample_rate = 48000
# плюсы там где нужно ставить ударения
example_text = 'В недрах тундры выдры в г+етрах т+ырят в вёдра ядра к+едров.'

# итерация по доступным голосам
#for speaker in model.speakers[:-1]:
# преобразовать текст в речь
audio = model.apply_tts(text=example_text, speaker=model.speakers[5], sample_rate=sample_rate)

# отобразить аудио в колаб
display(Audio(audio, rate=sample_rate))

Generated new voice


CPU times: user 4.36 s, sys: 280 ms, total: 4.64 s
Wall time: 5.16 s


In [ ]:
!pip install pydub

In [ ]:
from pydub import AudioSegment
from aiogram.types import FSInputFile, URLInputFile, BufferedInputFile
import io
import torchaudio

In [ ]:
import nest_asyncio
nest_asyncio.apply()

import os
import speech_recognition as sr
import asyncio
import logging
import sys
from os import getenv

from aiogram import Bot, Dispatcher, Router, F
from aiogram.enums import ParseMode
from aiogram.filters import CommandStart
from aiogram.types import Message

TOKEN = "TOKEN"

bot = Bot(TOKEN)
dp = Dispatcher()

@dp.message()
async def echo_handler(message: types.Message):
    if message.content_type == types.ContentType.VOICE:
        voice_path = "voice.ogg"
        await bot.download(message.voice, destination=voice_path)

        os.system('ffmpeg -y -i voice.ogg voice.wav')

        # Создание объекта Recognizer из библиотеки SpeechRecognition
        recognizer = sr.Recognizer()

        # Распознавание речи в голосовом сообщении
        try:
            with sr.AudioFile('voice.wav') as source:
                audio_data = recognizer.record(source)
                text = recognizer.recognize_google(audio_data, language="ru-RU")  #  нужный язык
                await message.reply(text, parse_mode=ParseMode.MARKDOWN)
        except sr.UnknownValueError:
            await message.reply("Не удалось распознать речь в голосовом сообщении.")
        except sr.RequestError as e:
            await message.reply(f"Ошибка при запросе к сервису распознавания речи: {str(e)}")

    elif message.content_type == types.ContentType.VIDEO_NOTE:
      video_path = 'video.mp4'
      await bot.download(message.video_note, destination=video_path)

      os.system('ffmpeg -y -i video.mp4 video.wav')

      recognizer = sr.Recognizer()

      try:
        with sr.AudioFile('video.wav') as source:
          audio_data = recognizer.record(source)
          text = recognizer.recognize_google(audio_data, language="ru-RU")  #  нужный язык
          await message.reply(text, parse_mode=ParseMode.MARKDOWN)
      except sr.UnknownValueError:
        await message.reply("Не удалось распознать речь в видео.")
      except sr.RequestError as e:
        await message.reply(f"Ошибка при запросе к сервису распознавания речи: {str(e)}")


    elif message.content_type == types.ContentType.TEXT:
      text = message.text
      audio = model.apply_tts(text=text,
                              speaker=model.speakers[5],
                              sample_rate=8000)

      save_path = 'audio.wav'
      torchaudio.save(save_path, audio.unsqueeze(0), sample_rate=8000)

        # аудио в формат .ogg
      os.system('ffmpeg -y -i audio.wav audio.mp3')

      audio_file = FSInputFile('audio.mp3')
      await bot.send_audio(chat_id=message.from_user.id, audio=audio_file)


async def main() -> None:
  await dp.start_polling(bot, skip_updates=True)


asyncio.run(main())

### Разделение на типы

In [ ]:
from aiogram import F

In [ ]:
import logging
import speech_recognition as sr
from aiogram import Bot, Dispatcher, types

import asyncio
import logging
import sys
from os import getenv
import os

from aiogram import Bot, Dispatcher, Router, types, F
from aiogram.enums import ParseMode
from aiogram.filters import CommandStart
from aiogram.types import Message
from aiogram.utils.markdown import hbold

import nest_asyncio

from aiogram.types import FSInputFile, URLInputFile, BufferedInputFile
import io
import torchaudio

import torch

In [ ]:
async def voice_to_text():
  recognizer = sr.Recognizer()
  # Распознавание речи в голосовом сообщении
  try:
    with sr.AudioFile('voice.wav') as source:
      audio_data = recognizer.record(source)
      text = recognizer.recognize_google(audio_data, language="ru-RU")  #  нужный язык
      return (text)
  except sr.UnknownValueError:
    return("Не удалось распознать речь")
  except sr.RequestError as e:
    return(f"Ошибка при запросе к сервису распознавания речи: {str(e)}")

In [ ]:
async def text_to_voice(text: str):
  language = 'ru'
  model_id = 'v3_1_ru'

  # загрузка модели с torchhub
  model, example_text = torch.hub.load(repo_or_dir='snakers4/silero-models',
                                       model='silero_tts',
                                       language=language,
                                       speaker=model_id,
                                       trust_repo=True)
  # частота дискретизации
  sample_rate = 8000
  audio = model.apply_tts(text=text,
                          speaker=model.speakers[5],
                          sample_rate=8000)
  save_path = 'audio.wav'
  torchaudio.save(save_path, audio.unsqueeze(0), sample_rate=8000)

  # аудио в формат .mp3
  os.system('ffmpeg -y -i audio.wav audio.mp3')
  audio_file = FSInputFile('audio.mp3')
  return audio_file

In [ ]:
TOKEN = "TOKEN"
bot = Bot(TOKEN)
dp = Dispatcher()

In [ ]:
@dp.message(F.voice)
async def from_voice_(message: types.Message):
  voice_path = "voice.ogg"
  await bot.download(message.voice, destination=voice_path)
  os.system('ffmpeg -y -i voice.ogg voice.wav')
  await message.reply(await voice_to_text(), parse_mode=ParseMode.MARKDOWN)


@dp.message(F.video_note)
async def from_video(message: types.Message):
  video_path = 'video.mp4'
  await bot.download(message.video_note, destination=video_path)
  os.system('ffmpeg -y -i video.mp4 voice.wav')
  await message.reply(await voice_to_text(), parse_mode=ParseMode.MARKDOWN)


@dp.message(F.text)
async def from_text(message: types.Message):
  result = await text_to_voice(message.text)
  await bot.send_audio(chat_id=message.from_user.id,
                           audio=result)

In [ ]:
async def main() -> None:
  await dp.start_polling(bot, skip_updates=True)

In [ ]:
asyncio.run(main())

## requirements.txt

In [ ]:
!pip freeze > requirements.txt

## Код для контейнера (хоста)

In [ ]:
import speech_recognition as sr
from aiogram import Bot, Dispatcher, types
import soundfile as sf

import asyncio
import logging
import sys
from os import getenv


from aiogram import Bot, Dispatcher, Router, types, F
from aiogram.enums import ParseMode
from aiogram.filters import CommandStart
from aiogram.types import Message
from aiogram.utils.markdown import hbold


from aiogram.types import FSInputFile, URLInputFile, BufferedInputFile
import io
import torchaudio

import torch



async def voice_to_text():
    recognizer = sr.Recognizer()
  # Распознавание речи в голосовом сообщении
    try:
        with sr.AudioFile('voice.wav') as source:
            audio_data = recognizer.record(source)
            text = recognizer.recognize_google(audio_data, language="ru-RU")  #  нужный язык
            return (text)
    except sr.UnknownValueError:
        return("Не удалось распознать речь")
    except sr.RequestError as e:
        return(f"Ошибка при запросе к сервису распознавания речи: {str(e)}")


async def text_to_voice(text: str):
    language = 'ru'
    model_id = 'v3_1_ru'

    # загрузка модели с torchhub
    model, example_text = torch.hub.load(repo_or_dir='snakers4/silero-models',
                                       model='silero_tts',
                                       language=language,
                                       speaker=model_id,
                                       trust_repo=True)
    # частота дискретизации
    sample_rate = 8000
    audio = model.apply_tts(text=text,
                          speaker=model.speakers[5],
                          sample_rate=8000)
    save_path = 'audio.wav'
    #torchaudio.save(save_path, audio.unsqueeze(0), sample_rate=8000)
    sf.write(save_path, audio.numpy(), sample_rate)

    # аудио в формат .mp3
    os.system('ffmpeg -y -i audio.wav audio.mp3')
    audio_file = FSInputFile('audio.mp3')
    return audio_file


TOKEN = "TOKEN"
bot = Bot(TOKEN)
dp = Dispatcher()


@dp.message(F.voice)
async def from_voice_(message: types.Message):
    voice_path = "voice.ogg"
    await bot.download(message.voice, destination=voice_path)
    os.system('ffmpeg -y -i voice.ogg voice.wav')
    await message.reply(await voice_to_text(), parse_mode=ParseMode.MARKDOWN)


@dp.message(F.video_note)
async def from_video(message: types.Message):
    video_path = 'video.mp4'
    await bot.download(message.video_note, destination=video_path)
    os.system('ffmpeg -y -i video.mp4 voice.wav')
    await message.reply(await voice_to_text(), parse_mode=ParseMode.MARKDOWN)


@dp.message(F.text)
async def from_text(message: types.Message):
    result = await text_to_voice(message.text)
    await bot.send_audio(chat_id=message.from_user.id,
                           audio=result)


async def main() -> None:
    await dp.start_polling(bot, skip_updates=True)

In [ ]:
asyncio.run(main())

RuntimeError: ignored

In [ ]:
!pip install omegaconf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 6.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144554 sha256=179f226a30f3ecb40ad460c09f4a2ccf211388efe50a96d9eda684de6936e154
  Stored in directory: /root/.cache/pip/wheels/12/93/dd/1f6a127edc45659556564c5730f6d4e300888f4bca2d4c5a88
Successfully built antlr4-python3-runtime


In [ ]:
import nest_asyncio
nest_asyncio.apply()

In [ ]:
asyncio.run(main())

Using cache found in /root/.cache/torch/hub/snakers4_silero-models_master
100%|██████████| 59.0M/59.0M [00:02<00:00, 20.8MB/s]


Generated new voice


Using cache found in /root/.cache/torch/hub/snakers4_silero-models_master


Generated new voice


## Итоговый код хост

In [ ]:
import speech_recognition as sr
from aiogram import Bot, Dispatcher, types
import soundfile as sf

import asyncio
import logging
import sys
import os
from os import getenv


from aiogram import Bot, Dispatcher, Router, types, F
from aiogram.enums import ParseMode
from aiogram.filters import CommandStart
from aiogram.types import Message
from aiogram.utils.markdown import hbold


from aiogram.types import FSInputFile, URLInputFile, BufferedInputFile
import io
#import torchaudio

import torch



async def voice_to_text():
    recognizer = sr.Recognizer()
  # Распознавание речи в голосовом сообщении
    try:
        with sr.AudioFile('voice.wav') as source:
            audio_data = recognizer.record(source)
            text = recognizer.recognize_google(audio_data, language="ru-RU")  #  нужный язык
            return (text)
    except sr.UnknownValueError:
        return("Не удалось распознать речь")
    except sr.RequestError as e:
        return(f"Ошибка при запросе к сервису распознавания речи: {str(e)}")


async def text_to_voice(text: str):
    language = 'ru'
    model_id = 'v3_1_ru'

    # загрузка модели с torchhub
    model, example_text = torch.hub.load(repo_or_dir='snakers4/silero-models',
                                       model='silero_tts',
                                       language=language,
                                       speaker=model_id,
                                       trust_repo=True)
    # частота дискретизации
    sample_rate = 8000
    audio = model.apply_tts(text=text,
                          speaker=model.speakers[5],
                          sample_rate=8000)
    save_path = 'audio.wav'
    # torchaudio.save(save_path, audio.unsqueeze(0), sample_rate=8000)
    sf.write(save_path, audio.numpy(), sample_rate)


    # аудио в формат .mp3
    os.system('ffmpeg -y -i audio.wav audio.mp3')
    audio_file = FSInputFile('audio.mp3')
    return audio_file


TOKEN = "TOKEN"
bot = Bot(TOKEN)
dp = Dispatcher()


@dp.message(F.voice)
async def from_voice_(message: types.Message):
    voice_path = "voice.ogg"
    await bot.download(message.voice, destination=voice_path)
    os.system('ffmpeg -y -i voice.ogg voice.wav')
    await message.reply(await voice_to_text(), parse_mode=ParseMode.MARKDOWN)


@dp.message(F.video_note)
async def from_video(message: types.Message):
    video_path = 'video.mp4'
    await bot.download(message.video_note, destination=video_path)
    os.system('ffmpeg -y -i video.mp4 voice.wav')
    await message.reply(await voice_to_text(), parse_mode=ParseMode.MARKDOWN)


@dp.message(F.text)
async def from_text(message: types.Message):
    result = await text_to_voice(message.text)
    await bot.send_audio(chat_id=message.from_user.id,
                           audio=result)


async def main() -> None:
    await dp.start_polling(bot, skip_updates=True)


asyncio.run(main())